In [0]:
pip install cloudscraper pandas

In [0]:
pip install bs4

In [0]:
pip install investpy pandas

In [0]:
from pyspark.sql.functions import to_date, col, current_date
import cloudscraper
from bs4 import BeautifulSoup
import pandas as pd
from delta.tables import DeltaTable

In [0]:
def extrair_caixin_services_spark():
    url = "https://br.investing.com/economic-calendar/chinese-caixin-services-pmi-596"

    # cria scraper que contorna o Cloudflare
    scraper = cloudscraper.create_scraper()
    resp    = scraper.get(url)
    resp.raise_for_status()
    html    = resp.text

    # parse com BeautifulSoup e parser interno (não precisa de lxml)
    soup  = BeautifulSoup(html, "html.parser")
    # normalmente a tabela histórica tem essa classe, mas ajuste se necessário
    table = soup.find("table", {"class": "genTbl closedTbl historicalTbl"})
    if table is None:
        table = soup.find("table")

    # cabeçalho
    header = [th.get_text(strip=True) for th in table.find_all("tr")[0].find_all(["th","td"])]
    # índice das colunas que nos interessam
    idx = {
        "Lançamento": header.index("Lançamento"),
        "Atual":      header.index("Atual"),
        "Anterior":   header.index("Anterior"),
        "Projeção":   header.index("Projeção")
    }

    # extrai linhas
    data = []
    for row in table.find_all("tr")[1:]:
        cols = row.find_all("td")
        if not cols: 
            continue
        vals = [c.get_text(strip=True) for c in cols]
        data.append({
            "date":         vals[idx["Lançamento"]],
            "actual_state": vals[idx["Atual"]],
            "close":        vals[idx["Anterior"]],
            "forecast":     vals[idx["Projeção"]],
        })

    # monta pandas e converte pra Spark
    df = pd.DataFrame(data)
    spark = SparkSession.builder.getOrCreate()
    return spark.createDataFrame(df)

# 3) Rode e veja no Databricks
df_caixin_services = extrair_caixin_services_spark()
display(df_caixin_services)


In [0]:


def fetch_commodities_historico(
    page_url: str,
    start_date: str,
    end_date: str,
    curr_id: str,
    smlID: str
):
    # 1) prepara cloudscraper e headers
    scraper = cloudscraper.create_scraper(
        browser={"browser": "chrome", "platform": "windows"}
    )
    scraper.headers.update({
        "User-Agent":       "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Accept":           "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language":  "pt-BR,pt;q=0.9",
        "Referer":          page_url,
        "Origin":           "https://br.investing.com",
        "X-Requested-With": "XMLHttpRequest",
    })
    scraper.get(page_url).raise_for_status()

    # 2) POST no Ajax
    payload = {
        "curr_id":      curr_id,
        "smlID":        smlID,
        "header":       "Date,Último,Abertura,Máxima,Mínima,Vol.",
        "st_date":      start_date,   # ex: "31/03/2025"
        "end_date":     end_date,     # ex: "01/05/2025"
        "interval_sec": "Daily",
        "sort_col":     "date",
        "sort_ord":     "DESC"
    }
    resp = scraper.post(
        "https://br.investing.com/instruments/HistoricalDataAjax",
        data=payload
    )
    resp.raise_for_status()

    # 3) extrai a <table> do HTML
    soup = BeautifulSoup(resp.text, "html.parser")
    table = soup.find("table")
    if not table:
        raise RuntimeError("Tabela não encontrada na resposta Ajax")

    # 4) monta linhas CSV
    headers = [th.get_text(strip=True) for th in table.find_all("th")]
    csv_lines = [",".join(headers)]
    for tr in table.find_all("tr"):
        tds = tr.find_all("td")
        if not tds: continue
        raw = [td.get_text(strip=True) for td in tds]
        clean = []
        for i, v in enumerate(raw):
            if i == 0:
                clean.append(v.replace(".", "/"))  # Data dd/MM/yyyy
            else:
                clean.append(v.replace(".", "")
                               .replace(",", ".")
                               .replace("%", ""))
        csv_lines.append(",".join(clean))

    # 5) carrega no Spark
    spark = SparkSession.builder.appName("InvestingDataFetch").getOrCreate()
    rdd = spark.sparkContext.parallelize(csv_lines)
    df_raw = (spark.read
                 .option("header", True)
                 .option("inferSchema", True)
                 .csv(rdd))

    # 6) renomeia e converte tipos
    df = (df_raw
        .withColumnRenamed("Data",    "date_str")
        .withColumnRenamed("Último",  "close_str")
        .withColumnRenamed("Abertura","open_str")
        .withColumnRenamed("Máxima",  "high_str")
        .withColumnRenamed("Mínima",  "low_str")
        .withColumnRenamed("Vol.",    "volume_str")
        .select(
            to_date(col("date_str"), "dd/MM/yyyy").alias("date"),
            col("close_str").cast("double").alias("close"),
            col("open_str").cast("double").alias("open"),
            col("high_str").cast("double").alias("high"),
            col("low_str").cast("double").alias("low"),
            col("volume_str").cast("double").alias("volume"),
        )
    )

    return df


In [0]:
# chama a função com os valores que funcionaram melhor pra você
df = fetch_commodities_historico(
    page_url="https://br.investing.com/indices/bloomberg-commodity-historical-data",
    start_date="31/03/1991",
    end_date="01/05/2025",
    curr_id="944041",
    smlID="1179962"
)


df.display()


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col, lit
import cloudscraper
from bs4 import BeautifulSoup

def fetch_investing_historico(
    page_url: str,
    start_date: str,
    end_date: str,
    curr_id: str,
    smlID: str
):
    # ——— 1) prepara cloudscraper e headers ———
    scraper = cloudscraper.create_scraper(
        browser={"browser": "chrome", "platform": "windows"}
    )
    scraper.headers.update({
        "User-Agent":       "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Accept":           "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Referer":          page_url,
        "Origin":           "https://br.investing.com",
        "X-Requested-With": "XMLHttpRequest",
    })
    scraper.get(page_url).raise_for_status()

    # ——— 2) faz o POST Ajax para pegar a tabela ———
    ajax_url = "https://br.investing.com/instruments/HistoricalDataAjax"
    payload = {
        "curr_id":      curr_id,
        "smlID":        smlID,
        "header":       "Date,Último,Abertura,Máxima,Mínima,Vol.",
        "st_date":      start_date,
        "end_date":     end_date,
        "interval_sec": "Daily",
        "sort_col":     "date",
        "sort_ord":     "DESC"
    }
    resp = scraper.post(ajax_url, data=payload)
    resp.raise_for_status()

    # ——— 3) extrai o HTML da tabela ———
    soup = BeautifulSoup(resp.text, "html.parser")
    table = soup.find("table")
    if not table:
        raise RuntimeError("Não encontrou a <table> de histórico.")

    # ——— 4) monta linhas CSV de forma segura ———
    csv_lines = []
    headers = [th.get_text(strip=True) for th in table.find_all("th")]
    csv_lines.append(",".join(headers))

    for tr in table.find_all("tr"):
        tds = tr.find_all("td")
        if not tds:
            continue
        raw = [td.get_text(strip=True) for td in tds]
        clean = []
        for i, v in enumerate(raw):
            if i == 0:
                clean.append(v.replace(".", "/"))           # Data dd/MM/yyyy
            else:
                clean.append(
                    v.replace(".", "")                      # remove milhar
                     .replace(",", ".")                     # vírgula → ponto
                     .replace("%", "")                      # remove % (var%)
                )
        csv_lines.append(",".join(clean))

    # ——— 5) carrega no Spark ———
    spark = SparkSession.builder.appName("InvestingFetch").getOrCreate()
    rdd = spark.sparkContext.parallelize(csv_lines)
    df_raw = (
        spark.read
             .option("header", True)
             .option("inferSchema", True)
             .csv(rdd)
    )

    # ——— 6) renomeia as colunas que existem ———
    mappings = {
        "Data":    "date_str",
        "Último":  "close_str",
        "Abertura":"open_str",
        "Máxima":  "high_str",
        "Mínima":  "low_str",
    }
    # só adiciona Vol. se vier no raw
    if "Vol." in df_raw.columns:
        mappings["Vol."] = "volume_str"

    df1 = df_raw
    for old, new in mappings.items():
        df1 = df1.withColumnRenamed(old, new)

    # ——— 7) monta o select final dinamicamente ———
    select_exprs = [
        to_date(col("date_str"), "dd/MM/yyyy").alias("date"),
        col("close_str").cast("double").alias("close"),
        col("open_str").cast("double").alias("open"),
        col("high_str").cast("double").alias("high"),
        col("low_str").cast("double").alias("low"),
    ]

    if "volume_str" in df1.columns:
        select_exprs.append(col("volume_str").cast("double").alias("volume"))
    else:
        # cria uma coluna volume cheia de NULLs, para manter o schema consistente
        select_exprs.append(lit(None).cast("double").alias("volume"))

    df_final = df1.select(*select_exprs)
    return df_final


In [0]:
df_historico= fetch_investing_historico(
    page_url="https://br.investing.com/indices/bloomberg-commodity-historical-data",
    start_date="01/04/1991",
    end_date="02/05/2025",
    curr_id="8849", 
    smlID="300004"  
)



In [0]:
(
    df_historico.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("dtb_thera.thera_raw.historico")

)

In [0]:

df_currencies = fetch_investing_historico(
    page_url="https://br.investing.com/currencies/usd-cny",
    start_date="01/04/1991",
    end_date="02/05/2025",
    curr_id="8849", 
    smlID="300004"  
)



In [0]:


(
    df_currencies.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("dtb_thera.thera_raw.currencies")

)

In [0]:

(
    df_caixin_services.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("dtb_thera.thera_raw.caixin_services")

)